# UrbanFloodBench Baseline Workflow

This notebook is the team-friendly entry point for:

- loading UrbanFloodBench data
- building cleaned node-level features
- running persistence and random forest baselines
- inspecting validation metrics

The notebook calls reusable code from `src/urbanflood`, so the team can collaborate in notebooks without duplicating logic.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
shared_path = '/content/drive/Shareddrives'
print(os.listdir(shared_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[]


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

PosixPath('/')

In [ ]:
!git clone https://github.com/Hiro208/UrbanFlood.git

import sys
sys.path.append('/content/UrbanFlood/src')

import pandas as pd

from urbanflood.dataset import load_model_assets
from urbanflood.features import build_training_dataset, clean_training_dataset
from urbanflood.baselines import (
    event_based_split,
    get_feature_columns,
    run_persistence_baseline,
    run_random_forest_baseline,
    evaluate_baselines,
)

pd.set_option('display.max_columns', 200)
print("urbanflood import success")

Cloning into 'UrbanFlood'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 22 (delta 2), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 73.09 KiB | 3.85 MiB/s, done.
Resolving deltas: 100% (2/2), done.
urbanflood import success


## Paths And Debug Settings

In [ ]:

from pathlib import Path

DATA_DIR = Path('/content/drive/Shareddrives/UrbanFloodData')
MAX_EVENTS_PER_MODEL = 2
MAX_TRAIN_ROWS = 50000
LAG_STEPS = (1, 2, 3, 5, 10)
RAIN_WINDOWS = (3, 5, 10)

DATA_DIR

PosixPath('/content/drive/Shareddrives/UrbanFloodData')

## Build Training Dataset

In [ ]:
frames = []
for model_id in (1, 2):
    assets = load_model_assets(DATA_DIR, model_id=model_id, split='train')
    frame = build_training_dataset(
        assets,
        lag_steps=LAG_STEPS,
        rain_windows=RAIN_WINDOWS,
        max_events=MAX_EVENTS_PER_MODEL,
    )
    frame = clean_training_dataset(frame)
    print(f'model {model_id}: {len(frame):,} rows')
    frames.append(frame)

dataset = pd.concat(frames, ignore_index=True)
dataset.shape

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/Shareddrives/UrbanFloodData/Model_1/train/1d_nodes_static.csv'

In [ ]:
dataset.head()

,timestep,node_idx,water_level,node_aux_flow,model_id,event_id,node_type,position_x,position_y,depth,invert_elevation,surface_elevation,base_area,edge_in_flow_mean,edge_in_flow_max,edge_in_flow_min,edge_in_velocity_mean,edge_in_velocity_max,edge_in_velocity_min,edge_out_flow_mean,edge_out_flow_max,edge_out_flow_min,edge_out_velocity_mean,edge_out_velocity_max,edge_out_velocity_min,edge_upstream_water_level_mean,edge_upstream_water_level_max,edge_upstream_water_level_min,edge_downstream_water_level_mean,edge_downstream_water_level_max,edge_downstream_water_level_min,surface_paired_water_level_mean,surface_paired_water_level_max,surface_paired_water_level_min,surface_paired_rainfall_mean,surface_paired_rainfall_max,surface_paired_rainfall_min,surface_paired_water_volume_mean,surface_paired_water_volume_max,surface_paired_water_volume_min,effective_area,reference_elevation,rainfall,target_water_level,target_delta_water_level,delta_water_level,water_level_lag_1,delta_water_level_lag_1,rainfall_lag_1,water_level_lag_2,delta_water_level_lag_2,rainfall_lag_2,water_level_lag_3,delta_water_level_lag_3,rainfall_lag_3,water_level_lag_5,delta_water_level_lag_5,rainfall_lag_5,water_level_lag_10,delta_water_level_lag_10,rainfall_lag_10,rainfall_sum_3,rainfall_mean_3,rainfall_sum_5,rainfall_mean_5,rainfall_sum_10,rainfall_mean_10,cumulative_rainfall,timestep_index,water_level_above_reference,is_raining,water_level_to_area_ratio,water_volume,area,roughness,min_elevation,elevation,aspect,curvature,flow_accumulation,pipe_paired_water_level_mean,pipe_paired_water_level_max,pipe_paired_water_level_min
0,0.0,0,294.87430,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,16.837931,31.815710,1.860152,4.970807,6.481443,3.460171,33.611320,33.611320,33.611320,8.418276,8.418276,8.418276,296.187460,296.18870,296.18622,288.22095,288.22095,288.22095,300.070648,300.070648,300.070648,0.143333,0.143333,0.143333,1.180193,1.180193,1.180193,12.56,292.342,0.143333,294.98570,0.11140,0.00000,0.00000,0.00000,0.000000,0.00000,0.00000,0.000000,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.143333,0.143333,0.143333,0.143333,0.143333,0.143333,0.143333,0,2.53230,1,0.201616,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0,294.98570,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.313536,32.773335,1.853738,5.056079,6.676529,3.435629,34.621025,34.621025,34.621025,8.554404,8.554404,8.554404,296.282965,296.37643,296.18950,288.25427,288.25427,288.25427,300.070374,300.070374,300.070374,0.136667,0.136667,0.136667,1.178370,1.178370,1.178370,12.56,292.342,0.136667,295.00534,0.01964,0.11140,294.87430,0.00000,0.143333,0.00000,0.00000,0.000000,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.280000,0.140000,0.280000,0.140000,0.280000,0.140000,0.280000,1,2.64370,1,0.210486,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2.0,0,295.00534,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.397679,32.941055,1.854302,5.071252,6.710697,3.431807,34.790653,34.790653,34.790653,8.577585,8.577585,8.577585,296.300165,296.41030,296.19003,288.25964,288.25964,288.25964,300.070435,300.070435,300.070435,0.136667,0.136667,0.136667,1.178905,1.178905,1.178905,12.56,292.342,0.136667,295.02298,0.01764,0.01964,294.98570,0.11140,0.136667,294.87430,0.00000,0.143333,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.416667,0.138889,0.416667,0.138889,0.416667,0.138889,0.416667,2,2.66334,1,0.212049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,0,295.02298,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.553244,33.064080,2.042407,5.138633,6.735759,3.541506,35.096320,35.096320,35.096320,8.619795,8.619795,8.619795,296.327085,296.43850,296.21567,288.26910,288.26910,288.26910,300.070404,300.070404,300.070404,0.136667,0.136667,0.136667,1.178556,1.178556,1.178556,12.56,292.342,0.136667,295.02390,0.00092,0.01764,295.00534,0.01964,0.136667,294.98570,0.11140,0.136667,294.8743,0.0000,0.143333,0.0,0.0,0.0,0.0,0.0,0.0,0.410000,0.136667,0.553333,0.138333,0.553333,0.138333,0.553

## Event-Based Validation Split

In [ ]:
train_df, val_df = event_based_split(dataset, validation_fraction=0.2)
if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(MAX_TRAIN_ROWS, random_state=42)

feature_columns = get_feature_columns(
    train_df,
    drop_columns=('target_water_level', 'target_delta_water_level', 'timestamp'),
)

train_df.shape, val_df.shape, len(feature_columns)

((50000, 83), (1678920, 83), 76)

## Run Baselines

In [ ]:
persistence_pred = run_persistence_baseline(val_df)

rf_pred, rf_model = run_random_forest_baseline(
    train_df=train_df,
    val_df=val_df,
    feature_columns=feature_columns,
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

metrics = evaluate_baselines([persistence_pred, rf_pred])
metrics

,model_name,rmse,mae,num_rows
0,persistence,0.022982,0.008444,1678920
1,random_forest,0.025243,0.009622,1678920


## Inspect Predictions

In [ ]:
rf_pred.head()

,model_id,event_id,node_type,node_idx,timestep,target_water_level,prediction,model_name
0,1,2,1,0,0.0,292.80490,291.664905,random_forest
1,1,2,1,0,1.0,292.78700,292.275850,random_forest
2,1,2,1,0,2.0,292.77158,292.521654,random_forest
3,1,2,1,0,3.0,292.76025,292.490809,random_forest
4,1,2,1,0,4.0,292.75177,292.419370,random_forest


hdr

LSTM

In [ ]:
dataset = dataset.sort_values(['node_idx', 'timestep']).reset_index(drop=True)

split_idx = int(len(dataset) * 0.8)

train_df = dataset.iloc[:split_idx].copy()
val_df = dataset.iloc[split_idx:].copy()

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)

train_df: (1966240, 83)
val_df: (491561, 83)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

TARGET = 'target_water_level'
SEQ_LEN = 10

dynamic_features = [
    'water_level',
    'delta_water_level',
    'rainfall',
    'node_aux_flow',
    'edge_in_flow_mean',
    'edge_out_flow_mean',
    'surface_paired_rainfall_mean',
    'surface_paired_water_level_mean',
]

static_features = [
    'node_type',
    'position_x',
    'position_y',
    'depth',
    'invert_elevation',
    'surface_elevation',
    'base_area',
    'effective_area',
    'reference_elevation',
    'area',
    'roughness',
    'min_elevation',
    'elevation',
    'aspect',
    'curvature',
    'flow_accumulation',
]

In [ ]:
def make_sliding_windows(df, seq_len, dynamic_features, static_features, target_col):
    X_seq_list = []
    X_static_list = []
    y_list = []

    for node_id, g in df.groupby('node_idx'):
        g = g.sort_values('timestep').reset_index(drop=True)

        if len(g) <= seq_len:
            continue

        dyn = g[dynamic_features].values.astype(np.float32)
        sta = g[static_features].values.astype(np.float32)
        y = g[target_col].values.astype(np.float32)

        for i in range(seq_len, len(g)):
            X_seq_list.append(dyn[i-seq_len:i])
            X_static_list.append(sta[i])
            y_list.append(y[i])

    X_seq = np.array(X_seq_list, dtype=np.float32)
    X_static = np.array(X_static_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    return X_seq, X_static, y

In [ ]:
X_train_seq, X_train_static, y_train = make_sliding_windows(
    train_df,
    seq_len=SEQ_LEN,
    dynamic_features=dynamic_features,
    static_features=static_features,
    target_col=TARGET,
)

X_val_seq, X_val_static, y_val = make_sliding_windows(
    val_df,
    seq_len=SEQ_LEN,
    dynamic_features=dynamic_features,
    static_features=static_features,
    target_col=TARGET,
)

print("X_train_seq:", X_train_seq.shape)
print("X_train_static:", X_train_static.shape)
print("y_train:", y_train.shape)
print("X_val_seq:", X_val_seq.shape)
print("X_val_static:", X_val_static.shape)
print("y_val:", y_val.shape)

X_train_seq: (1934380, 10, 8)
X_train_static: (1934380, 16)
y_train: (1934380,)
X_val_seq: (480421, 10, 8)
X_val_static: (480421, 16)
y_val: (480421,)


In [ ]:
seq_scaler = StandardScaler()
static_scaler = StandardScaler()

n_train, seq_len, n_dyn = X_train_seq.shape
n_val = X_val_seq.shape[0]

X_train_seq_2d = X_train_seq.reshape(-1, n_dyn)
X_val_seq_2d = X_val_seq.reshape(-1, n_dyn)

X_train_seq_scaled = seq_scaler.fit_transform(X_train_seq_2d).reshape(n_train, seq_len, n_dyn)
X_val_seq_scaled = seq_scaler.transform(X_val_seq_2d).reshape(n_val, seq_len, n_dyn)

X_train_static_scaled = static_scaler.fit_transform(X_train_static)
X_val_static_scaled = static_scaler.transform(X_val_static)

print(X_train_seq_scaled.shape)
print(X_val_seq_scaled.shape)

(1934380, 10, 8)
(480421, 10, 8)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

seq_input = Input(shape=(X_train_seq_scaled.shape[1], X_train_seq_scaled.shape[2]), name='seq_input')
x_seq = LSTM(64, return_sequences=True)(seq_input)
x_seq = Dropout(0.2)(x_seq)
x_seq = LSTM(32)(x_seq)
x_seq = Dropout(0.2)(x_seq)

static_input = Input(shape=(X_train_static_scaled.shape[1],), name='static_input')
x_static = Dense(32, activation='relu')(static_input)
x_static = Dropout(0.2)(x_static)

x = Concatenate()([x_seq, x_static])
x = Dense(32, activation='relu')(x)
output = Dense(1)(x)

lstm_model = Model(inputs=[seq_input, static_input], outputs=output)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

early_stop_lstm = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    [X_train_seq_scaled, X_train_static_scaled],
    y_train,
    validation_data=([X_val_seq_scaled, X_val_static_scaled], y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop_lstm],
    verbose=1
)

Epoch 1/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 432s 28ms/step - loss: 746.2405 - mae: 11.6307 - val_loss: 220.8154 - val_mae: 2.9317
Epoch 2/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 456s 30ms/step - loss: 103.9784 - mae: 6.9223 - val_loss: 315.9231 - val_mae: 3.8237
Epoch 3/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 394s 26ms/step - loss: 14.3835 - mae: 2.5938 - val_loss: 125.2411 - val_mae: 2.0583
Epoch 4/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 438s 26ms/step - loss: 1.8006 - mae: 0.9173 - val_loss: 148.9467 - val_mae: 2.1658
Epoch 5/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 442s 26ms/step - loss: 1.1558 - mae: 0.7275 - val_loss: 218.9736 - val_mae: 2.1911
Epoch 6/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 390s 26ms/step - loss: 0.8583 - mae: 0.6043 - val_loss: 217.3379 - val_mae: 2.3532
Epoch 7/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 398s 26ms/step - loss: 0.7417 - mae: 0.5419 - val_loss: 196.6934 - val_mae: 2.0983
Epoch 8/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 434s 26ms/step - loss: 0.6999 - mae: 0.5155 - val_loss: 243.4

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lstm_pred = lstm_model.predict([X_val_seq_scaled, X_val_static_scaled], verbose=0).flatten()

lstm_metrics = pd.DataFrame([{
    'model_name': 'LSTM_window10',
    'rmse': np.sqrt(mean_squared_error(y_val, lstm_pred)),
    'mae': mean_absolute_error(y_val, lstm_pred),
    'r2': r2_score(y_val, lstm_pred),
}])

print(lstm_metrics)

      model_name       rmse       mae        r2
0  LSTM_window10  11.191121  2.058275  0.992418


CNN-LSTM

In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D

tf.keras.utils.set_random_seed(42)

seq_input_cnn = Input(shape=(X_train_seq_scaled.shape[1], X_train_seq_scaled.shape[2]), name='seq_input')

x_seq_cnn = Conv1D(filters=64, kernel_size=3, activation='relu', padding='causal')(seq_input_cnn)
x_seq_cnn = MaxPooling1D(pool_size=2)(x_seq_cnn)
x_seq_cnn = LSTM(32)(x_seq_cnn)
x_seq_cnn = Dropout(0.2)(x_seq_cnn)

static_input_cnn = Input(shape=(X_train_static_scaled.shape[1],), name='static_input')
x_static_cnn = Dense(32, activation='relu')(static_input_cnn)
x_static_cnn = Dropout(0.2)(x_static_cnn)

x_cnn = Concatenate()([x_seq_cnn, x_static_cnn])
x_cnn = Dense(32, activation='relu')(x_cnn)
output_cnn = Dense(1)(x_cnn)

cnn_lstm_model = Model(inputs=[seq_input_cnn, static_input_cnn], outputs=output_cnn)

cnn_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

early_stop_cnn = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history_cnn_lstm = cnn_lstm_model.fit(
    [X_train_seq_scaled, X_train_static_scaled],
    y_train,
    validation_data=([X_val_seq_scaled, X_val_static_scaled], y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop_cnn],
    verbose=1
)

Epoch 1/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 157s 10ms/step - loss: 673.3172 - mae: 11.1841 - val_loss: 196.8103 - val_mae: 3.2233
Epoch 2/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 152s 10ms/step - loss: 77.7323 - mae: 5.9206 - val_loss: 284.8474 - val_mae: 3.2221
Epoch 3/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 159s 11ms/step - loss: 5.3274 - mae: 1.5345 - val_loss: 135.5697 - val_mae: 1.4994
Epoch 4/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 151s 10ms/step - loss: 1.7710 - mae: 0.8606 - val_loss: 171.1698 - val_mae: 1.5780
Epoch 5/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 152s 10ms/step - loss: 1.4849 - mae: 0.7982 - val_loss: 282.5118 - val_mae: 2.1974
Epoch 6/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 155s 10ms/step - loss: 1.4430 - mae: 0.7679 - val_loss: 249.5425 - val_mae: 2.0342
Epoch 7/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 149s 10ms/step - loss: 1.4037 - mae: 0.7496 - val_loss: 342.3151 - val_mae: 2.0436
Epoch 8/30
15113/15113 ━━━━━━━━━━━━━━━━━━━━ 211s 10ms/step - loss: 1.3471 - mae: 0.7341 - val_loss: 295.507

In [ ]:
cnn_lstm_pred = cnn_lstm_model.predict([X_val_seq_scaled, X_val_static_scaled], verbose=0).flatten()

cnn_lstm_metrics = pd.DataFrame([{
    'model_name': 'CNN-LSTM_window10',
    'rmse': np.sqrt(mean_squared_error(y_val, cnn_lstm_pred)),
    'mae': mean_absolute_error(y_val, cnn_lstm_pred),
    'r2': r2_score(y_val, cnn_lstm_pred),
}])

print(cnn_lstm_metrics)

          model_name       rmse       mae        r2
0  CNN-LSTM_window10  11.643439  1.499431  0.991793
